## Pre-processing Notebook

This notebook contains instructions on how to access and download the data from the source, cropping the data for a given watershed boundary, calculating the mode (most frequent pixels) using Numba and Dask, reprojections, and Kronecker product to get Landsat and Sentinel datasets ready for comparison. 

*Note: These instructions are written for the Colorado River Basin specifically but can be applied elsewhere with a new changes on area of interest when downloading.*

## Downloading the Data 

1. Yearly Surface Water History Data, 2015 - 2021
2. Dynamic World Global Surface Water Data, 2015 - 2023
3. USGS NHDPlus Flowlines
4. Watershed Boundary Dataset
5. Palmer Drought Severity Index Data, 2015 - 2023
6. Precipitation Data, 2015 - 2023

### 1. Yearly Surface Water History Data, 2015 - 2021

This dataset is part of the [Global Suface Water Dataset](https://global-surface-water.appspot.com/) from Pekel etal. 2016 *High-resolution mapping of global surface water and its long-term changes* ([DOI:10.1038/nature20584](https://www.nature.com/articles/nature20584)). 

The Yearly Water History Dataset contains annual classification of 30m pixels into four classifications – permanent water (pixel value - 3), seasonal water (2), not water (1), and no observation (0) from 1984-2021. Note for this study, we only used years 2015 through 2021.  

Full detail description on how to download the tiles and processing them are located [here](https://github.com/ppuentex/Surface_Water_Analysis-CRB/blob/main/Codes/pre-processing.ipynb), this is a notebook I wrote for a separate project. 

**The final version of these files can be found in the folder `data/landsat-yearly/`**

### 2. Dynamic World Global Surface Water Data, 2015 - 2023

This dataset is publically available and developed my the Hydrology & Hydroinformatics innovation Lab at the University of Texas at Arlington, United States. Find more information [here](https://www.hydroshare.org/resource/9d60389f55b648149a788a2ff7bc3766/) on how to download. The use of Google Earth Engine is necessary to download. 

Navigate to Google Earth Engine and use the code snippets below to download the tiles for your area of interest. 

```
var global_map = ee.ImageCollection('projects/h2i-lab/assets/DynamicWorld_v1/Main/GlobalGrids_10m')
// Convert ImageCollection to a list
var imageCount = global_map.size();
var imageList = global_map.toList(imageCount);

//Create the geometry for your area of interest
var westernUS = ee.Geometry.Rectangle([-110, 38, -100, 44]);

//define projection settings 
var projection = global_map.first().select(0).projection();

print(projection) //returns EPSG 4326

// Apply EPSG 4326 using setDefaultProjection to each image in the ImageCollection
var projectedImages = global_map.map(function(image) {
  return image.setDefaultProjection(projection);
});

// Want to filter only the images (tiles) within our area of interest bounds
var filteredDataset = projectedImages.filterBounds(westernUS);

// Create a FeatureCollection of the tile footprints
var tileFootprints = filteredDataset.map(function(image) {
  return ee.Feature(image.geometry(), {
    id_no: image.get('id_no')
  });
});

// visualize tile boundaries
//Map.addLayer(tileFootprints, {color: 'red'}, 'Tile Footprints');
// 3. Convert to FeatureCollection for display
var footprintCollection = ee.FeatureCollection(tileFootprints);

// 4. Add all tile footprints to the map
Map.addLayer(footprintCollection.style({color: 'red', fillColor: '00000000'}), {}, 'All Tile Footprints');

// Optional: Center map on your geometry
Map.centerObject(westernUS, 6);
Map.addLayer(westernUS, {color: 'blue'}, 'WesternUS');

```

The output of the code above should look like the screenshot below, where the blue polygon is the area of interest and the red tiles are the tiles that overlap the area of interest and will be the tiles needed to download to complete the area of interest. 


<img src="./images/tiles.png" alt="Alt Text" width = 334 height = 444>


The code below will visualize the surface extent, water occurrence, or water variability. You will need to uncomment based on what band you are interested in visualizing. 

**Note: When downloaded, the tiles will have all three bands included.** 

```
filteredDataset.evaluate(function(col) {
  col.features.forEach(function(imgInfo, index) {
    var imgId = imgInfo.properties.id_no;
    var image = ee.Image(imgInfo.id);

    // Visualization parameters
    var vis1 = {min: 1, max: 3, palette: ['blue', 'green', 'orange']};
    //var vis2 = {min: 0, max: 100, palette: ['white', 'blue']};
    //var vis3 = {min: 0, max: 100, palette: ['white', 'red']};

    // Add layers to map with ID label
    Map.addLayer(image.select('b1'), vis1, 'surface extent: ' + imgId);
    //Map.addLayer(image.select('b2'), vis2, 'water occurrence: ' + imgId);
    //Map.addLayer(image.select('b3'), vis3, 'water variability: ' + imgId);
  });

  // Optionally center the map on your geometry
  Map.centerObject(westernUS, 6);
});
```

The output should look like the screenshot below. This example is the surface extent, showing permanent water in blue, seasonal water in green, and land in orange. 

<img src="./images/extent.png" alt="Alt Text" width = 334 height = 444>

Finally to download desired AOI tiles, run the code below. This code will save tiles in your Google Drive in a folder called `EarthEngineExports`.

```
// Retrieve ID's from tiles
var idList = filteredDataset.aggregate_array('id_no'); 
print(idList); // Prints out the list of id's you should expect to export for your AOI

// BATCH EXPORTATION
var batchSize = 10; // Change to how many appear in your list


idList.evaluate(function(ids) {  // Use .evaluate() to access list in JavaScript
  ids.slice(0,batchSize).forEach(function(id_no, index) {  // Process batch
    var image0 = filteredDataset.filter(ee.Filter.eq('id_no', id_no));
    var image1 = image0.first().select(['b1', 'b2', 'b3']); // modify for band selection
    
    Export.image.toDrive({
      image: image1,
      description: 'Exported_' + id_no,
      folder: 'EarthEngineExports',
      fileNamePrefix: id_no + '_SWextent_occur_var',
      scale: 10,
      region: image1.geometry(),
      maxPixels: 1e9,
      crs: 'EPSG:4326',
      fileFormat: 'GeoTIFF'
    });

    //Print info before exporting to make sure we are extracting what is needed 
    // print('Processing ID:', id_no);
    // print('Image:', image1);
    // print('Bands:', image1.bandNames());
    // print('Projection:', image1.projection());
    // print('Scale:', image1.projection().nominalScale());
   });
 });
 ```

### 3. USGS NHDPlus Flowlines

These are instructions to download the National Hydrography Dataset (NHD) Plus. Find more information about the dataset [here](https://www.usgs.gov/national-hydrography/nhdplus-high-resolution). Here are the steps I followed to get the shapefiles of the flowlines with their associated stream order for all HUC4's in the CRB. 

**This code is specifically written to download the NHDflowline which contains geometry of all flowlines and the NHDPlusFlowlineVAA which contains the stream order values for each flowline. This is ONLY for the CRB. But instructions can be applied to any other region.**

Download to saving the files needed: 
1. Navigate to [The National Map](https://prd-tnm.s3.amazonaws.com/index.html?prefix=StagedProducts/Hydrography/NHDPlusHR/VPU/Current/GPKG/). Download the `zip` file for your HUC4 of choice `NHDPLUS_H_<HUC4-ID>_HU4_20220414_GPKG.zip`. Here we download 1401, 1402, ..., 1408 for the Upper CRB, and 1501, 1502, ... 1507 for the Lower CRB. 
2. Once downloaded, the zip file will contain the `.gpkg` file. This will need to be read in Python. 
3. Merge the stream orders from NHDPlusFlowlines VAA to the NHDFlowlines
4. Reproject to ESPG 3857 (to match projection of raster files)
5. Save the information as `HUC4_<HUC4-ID>_flowlines.shp`. 

Below is an example code for this process for one HUC4. Note the `.gpkg` file is not included here due to its size. Repeat this workflow for any additional HUC4. </br>
**The final version used in the analysis here can be found in `data/zenodo-data/crb_huc4_stream_flowlines/`.**

In [ ]:
#this cell is necessary to not get any errors when loading shapefiles 
import os
os.environ["PROJ_DATA"] = "/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/fiona/proj_data"
# replace the /Users/ppuente/github/detection_comparison-CRB/comparison-env to your own path and what you named the your environment (ie. comparison-env)

In [18]:
import geopandas as gpd 

geopackage_path = '../data/NHDPLUS_H_1401_HU4_20220414_GPKG/NHDPLUS_H_1401_HU4_20220414_GPKG.gpkg'

#list all layers in the GeoPackage just to see what it contains
#layer_info = gpd.list_layers(geopackage_path)

NHD_flowlines = gpd.read_file(geopackage_path, layer = 'NHDFlowline')
NHDPlus_flowlines = gpd.read_file(geopackage_path, layer = 'NHDPlusFlowlineVAA')

/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(


In [20]:
#print the columns in each layer

#print('columns in NHDFlowlines', NHD_flowlines.columns)

#print("columns in NHDPlusFlowlineVAA",NHDPlus_flowlines.columns)

There are different types of flowlines that can be found [here](https://www.usgs.gov/ngp-standards-and-specifications/national-hydrography-dataset-nhd-data-dictionary-feature-classes). For this study's case, the flowline type (`ftype`) we remove are 428, 334, and 420. 

In [ ]:
ftype_vals = [428, 334, 420]
#removing all rows that contain the ftype in list
NHD_flowlines = NHD_flowlines[~NHD_flowlines['ftype'].isin(ftype_vals)]

Now, with the flowline types being filtered, we will merge the stream orders from the NHDPlus_Flowlines with the NHD_Flowlines on the `nhdplusid`. Note that you can choose to merge anything in the NHD_Flowlines (ie. streamlevel, streamcalc, etc) but for our study we only needed streamorder. 

In [ ]:
selected_flowlines = NHDPlus_flowlines[['nhdplusid', 'streamorde']] #getting the id and streamorder 

In [24]:
# Merge while keeping only matching rows (inner join)
merged_flowlines = NHD_flowlines.merge(selected_flowlines, on = 'nhdplusid', how = 'inner')

In [25]:
print("Original NHDFlowlines size:", NHD_flowlines.shape)
print("Original Selected NHDPlusFlowlines size:", selected_flowlines.shape)
print("Merged dataset size:", merged_flowlines.shape)

Original NHDFlowlines size: (134569, 25)
Original Selected NHDPlusFlowlines size: (134241, 2)
Merged dataset size: (133517, 26)


In [ ]:
#number of flowline segments per stream orders
print(merged_flowlines['streamorde'].value_counts())

streamorde
1    67983
2    31896
3    16610
4     8755
5     3932
6     1769
7     1038
9      882
8      652
Name: count, dtype: int64


In [28]:
#reprojection prior to saving
merged_flowlines = merged_flowlines.to_crs(epsg=3857)
print("Old CRS:", NHD_flowlines.crs)
print("New CRS:", merged_flowlines.crs)

Old CRS: EPSG:4269
New CRS: EPSG:3857


In [ ]:
#save as a shapefile 
#merged_flowlines.to_file("./data/zenodo-data/crb_huc4_stream_flowlines/huc4_1407_flowlines.shp", driver="ESRI Shapefile")